# ChimeraDB: Industrial IoT Example

**Smart Building Heating Monitoring with Knowledge Graphs**

This notebook demonstrates how to use ChimeraDB for industrial IoT applications:
- **Knowledge Graph**: Metadata about buildings, rooms, and sensors (EMBEDDED)
- **Timeseries Data**: Actual power readings over time (NOT EMBEDDED)
- **SQL Joins**: Combine both for intelligent analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codimusmaximus/chimeradb/blob/main/examples/industrial_iot_colab.ipynb)

---

## Installation

Install ChimeraDB and dependencies:

In [ ]:
!pip install -q chimeradb duckdb
print("✓ ChimeraDB installed successfully!")

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from chimeradb import KnowledgeGraph
import random
from datetime import datetime, timedelta

print("✓ Imports complete")

## The Problem

**User Question**: *"Which rooms in Building A are using too much heating?"*

To answer this, an LLM needs:
1. **Knowledge Graph** (embedded): Buildings, rooms, sensors, baselines
2. **Timeseries Data** (not embedded): Actual power readings over time
3. **SQL Analytics**: Join them together to compare actual vs. expected usage

---

## Step 1: Build Knowledge Graph (Metadata with Embeddings)

In [ ]:
# Create knowledge graph
kg = KnowledgeGraph(":memory:")
print("✓ Created knowledge graph")

In [ ]:
# Add Building A
kg.add_entity(
    "building_a",
    {
        "name": "Building A",
        "type": "Commercial Office",
        "description": "Modern office building with mixed-use spaces including offices, server rooms, and parking"
    },
    ["Building"],
    embed_field="description"
)
print("✓ Added Building A")

In [ ]:
# Add Rooms with heating baselines
rooms = [
    {"id": "room_office_201", "name": "Office 201", "room_type": "Office", "baseline_watts": 50,
     "description": "Standard office space with workstation and natural lighting"},
    {"id": "room_office_202", "name": "Office 202", "room_type": "Office", "baseline_watts": 60,
     "description": "Corner office with large windows and meeting area"},
    {"id": "room_server", "name": "Server Room", "room_type": "Technical", "baseline_watts": 0,
     "description": "Climate-controlled server room with precision cooling system"},
    {"id": "room_garage", "name": "Underground Garage", "room_type": "Parking", "baseline_watts": 0,
     "description": "Underground parking garage with vehicle access"},
    {"id": "room_conference", "name": "Conference Room A", "room_type": "Meeting", "baseline_watts": 100,
     "description": "Large conference room with AV equipment and seating for 20"}
]

for room in rooms:
    room_id = room.pop("id")
    kg.add_entity(room_id, room, ["Room"], embed_field="description")
    kg.add_relationship("building_a", room_id, "CONTAINS")
    print(f"  ✓ Added {room['name']} (baseline: {room['baseline_watts']}W)")

In [ ]:
# Add Power Sensors
sensors = [
    {"id": "sensor_pwr_201", "name": "PWR-201-A", "room_id": "room_office_201",
     "description": "Power consumption meter for office heating system"},
    {"id": "sensor_pwr_202", "name": "PWR-202-A", "room_id": "room_office_202",
     "description": "Power consumption meter for corner office HVAC"},
    {"id": "sensor_pwr_server", "name": "PWR-SRV-01", "room_id": "room_server",
     "description": "Power meter for server room cooling system"},
    {"id": "sensor_pwr_garage", "name": "PWR-GAR-01", "room_id": "room_garage",
     "description": "Power meter for garage ventilation and lighting"},
    {"id": "sensor_pwr_conf", "name": "PWR-CONF-A", "room_id": "room_conference",
     "description": "Power meter for conference room climate control"}
]

for sensor in sensors:
    sensor_id = sensor.pop("id")
    room_id = sensor.pop("room_id")
    kg.add_entity(sensor_id, sensor, ["Sensor", "PowerMeter"], embed_field="description")
    kg.add_relationship(room_id, sensor_id, "MONITORED_BY")
    print(f"  ✓ Added {sensor['name']}")

print("\n✓ Knowledge graph complete: 1 building, 5 rooms, 5 sensors")

## Step 2: Add Timeseries Data (No Embeddings)

**Key Point**: Timeseries data is stored in a separate table and NOT embedded. This saves computation and storage.

In [ ]:
# Create separate table for timeseries data
kg.conn.execute("""
    CREATE TABLE power_readings (
        sensor_id VARCHAR,
        timestamp TIMESTAMP,
        power_watts FLOAT,
        temperature_celsius FLOAT
    )
""")
print("✓ Created power_readings table (separate from knowledge graph)")

In [ ]:
# Generate simulated timeseries data for 7 days
sensor_configs = {
    'sensor_pwr_201': {'mean': 95, 'std': 10},      # OVERUSE
    'sensor_pwr_202': {'mean': 55, 'std': 8},       # OK
    'sensor_pwr_server': {'mean': 5, 'std': 2},     # OK
    'sensor_pwr_garage': {'mean': 2, 'std': 1},     # OK
    'sensor_pwr_conf': {'mean': 145, 'std': 15}     # OVERUSE
}

base_time = datetime.now() - timedelta(days=7)
readings = []

for sensor_id, config in sensor_configs.items():
    for hour in range(7 * 24):  # 168 hours = 7 days
        timestamp = base_time + timedelta(hours=hour)
        power = max(0, random.gauss(config['mean'], config['std']))
        temp = random.gauss(21, 2)
        readings.append((sensor_id, timestamp, power, temp))

kg.conn.executemany("""
    INSERT INTO power_readings (sensor_id, timestamp, power_watts, temperature_celsius)
    VALUES (?, ?, ?, ?)
""", readings)

print(f"✓ Inserted {len(readings)} timeseries readings (NOT embedded)")
print(f"  → {len(sensor_configs)} sensors × 168 hours = {len(readings)} data points")
print("  → Only metadata is embedded, not the timeseries data!")

## Step 3: LLM Reasoning Workflow

### Step 3.1: RAG - Search Knowledge Graph

In [ ]:
# LLM searches for relevant concepts
concepts = kg.search("heating power consumption room sensor monitoring", top_k=5)

print("Found relevant entities:\n")
for i, entity in enumerate(concepts, 1):
    name = entity['properties'].get('name', entity['id'])
    labels = ', '.join(entity.get('labels', []))
    similarity = entity['similarity']
    print(f"  {i}. {name:30s} [{labels:20s}] (similarity: {similarity:.3f})")

print("\n✓ LLM now knows: Buildings → Rooms → Sensors → Power")

### Step 3.2: Graph Traversal - Find Specific Entities

In [ ]:
# Traverse from Building A to find all rooms
building_rooms = kg.traverse("building_a", direction="outgoing", relation_type="CONTAINS")

print(f"Building A contains {len(building_rooms)} rooms:\n")
for room in building_rooms:
    print(f"  - {room['properties']['name']}")

# For each room, find sensors
print("\nRoom → Sensor mappings:\n")
for room in building_rooms:
    sensors = kg.traverse(room['id'], direction="outgoing", relation_type="MONITORED_BY")
    baseline = room['properties']['baseline_watts']
    for sensor in sensors:
        print(f"  {room['properties']['name']:25s} → {sensor['properties']['name']:15s} (baseline: {baseline}W)")

### Step 3.3: SQL Analytics - Join Knowledge Graph with Timeseries

**This is the key**: Join embedded metadata (knowledge graph) with non-embedded timeseries data using SQL.

In [ ]:
# Join knowledge graph with timeseries data
results = kg.query("""
    WITH sensor_averages AS (
        SELECT
            sensor_id,
            AVG(power_watts) as avg_power,
            MAX(power_watts) as max_power
        FROM power_readings
        WHERE timestamp >= NOW() - INTERVAL '7 days'
        GROUP BY sensor_id
    )
    SELECT
        json_extract_string(room.properties, 'name') as room_name,
        CAST(json_extract_string(room.properties, 'baseline_watts') AS INTEGER) as baseline,
        CAST(sa.avg_power AS INTEGER) as current_avg,
        CAST(sa.max_power AS INTEGER) as max_reading
    FROM nodes room
    JOIN edges e ON e.from_id = room.id AND e.edge_type = 'MONITORED_BY'
    JOIN nodes sensor ON sensor.id = e.to_id
    JOIN sensor_averages sa ON sa.sensor_id = sensor.id
    WHERE room.labels LIKE '%Room%'
    ORDER BY current_avg DESC
""")

print("Room Power Analysis (7-day average):\n")
overuse_rooms = []

for room_name, baseline, current_avg, max_reading in results:
    excess = current_avg - baseline
    status = "⚠️  OVERUSE" if excess > baseline * 0.1 and baseline > 0 else "✓ OK"
    print(f"  {room_name:25s}: {current_avg:3d}W (baseline: {baseline:3d}W, max: {max_reading:3d}W) [{status}]")
    
    if excess > baseline * 0.1 and baseline > 0:
        overuse_rooms.append({
            'room': room_name,
            'current': current_avg,
            'baseline': baseline,
            'excess': excess
        })

## Step 4: LLM Answer

In [ ]:
if overuse_rooms:
    print(f"\n🔍 Found {len(overuse_rooms)} room(s) with excessive heating:\n")
    for room_info in overuse_rooms:
        print(f"  • {room_info['room']}:")
        print(f"      Current:  {room_info['current']}W")
        print(f"      Baseline: {room_info['baseline']}W")
        print(f"      Excess:   +{room_info['excess']}W ({room_info['excess']/room_info['baseline']*100:.0f}% over)")
        print()
    
    print("💡 Recommendations:")
    print("  - Check thermostat settings")
    print("  - Verify window/door seals")
    print("  - Consider HVAC maintenance")
else:
    print("\n✓ All rooms within expected baselines")

## Data Architecture Summary

In [ ]:
node_count = kg.query("SELECT COUNT(*) FROM nodes")[0][0]
reading_count = kg.query("SELECT COUNT(*) FROM power_readings")[0][0]

print("="*70)
print("DATA ARCHITECTURE")
print("="*70)
print("\nWhat gets EMBEDDED (knowledge graph):")
print("  ✓ Buildings, Rooms, Sensors (metadata)")
print("  ✓ Descriptions, baselines, configurations")
print(f"  → {node_count} entities WITH embeddings")

print("\nWhat does NOT get embedded (timeseries):")
print("  ✓ Power readings over time")
print("  ✓ Stored in separate table")
print(f"  → {reading_count} readings WITHOUT embeddings")

print("\nHow they connect:")
print("  • SQL join using sensor_id as foreign key")
print("  • Knowledge graph provides context")
print("  • Timeseries provides measurements")
print("  • Best of both worlds: Semantic search + Fast analytics")
print("="*70)

## Key Takeaways

1. **LLM Reasoning**: RAG (search) → Graph (traverse) → SQL (analyze)
2. **Data Architecture**: Embed metadata, don't embed timeseries, join with SQL
3. **Storage Efficiency**: 11 entities with embeddings vs. 840 readings without
4. **Production Pattern**: Knowledge graphs provide context, timeseries provides data

---

## Learn More

- **GitHub**: [github.com/codimusmaximus/chimeradb](https://github.com/codimusmaximus/chimeradb)
- **Documentation**: [ChimeraDB Docs](https://github.com/codimusmaximus/chimeradb/tree/main/docs)
- **More Examples**: [ChimeraDB Examples](https://github.com/codimusmaximus/chimeradb/tree/main/examples)

In [ ]:
# Cleanup
kg.close()
print("✅ Example complete!")